# MentalBERT baseline — Google Colab runner

Fine-tunes the naive multi-class MentalBERT baseline (`src/modeling/train.py`) and
produces held-out softmax predictions (`src/modeling/predict.py`).

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type →
Hardware accelerator → GPU (T4 is fine)*.

**One-time Drive setup.** Put your data and outputs on Google Drive so they survive
runtime resets. Create this layout in *MyDrive* (upload your local `Datasets/` into it):
```
MyDrive/mental-health-fyp/
    Datasets/            <- upload your local Datasets/ here (DAIC-WOZ/, RedditMentalHealth/)
    Models/              <- created automatically for checkpoints/splits/predictions
```
The same `src/modeling` code runs here and locally — only the `--data-root` /
`--artifacts-root` paths differ.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this if you named your Drive folder differently.
DRIVE_ROOT = '/content/drive/MyDrive/mental-health-fyp'
DATA_ROOT = f'{DRIVE_ROOT}/Datasets'
ARTIFACTS_ROOT = f'{DRIVE_ROOT}/Models'

import os
assert os.path.isdir(DATA_ROOT), f'Datasets not found at {DATA_ROOT} — upload them to Drive first.'
print('Datasets found:', os.listdir(DATA_ROOT))

## 3. Clone the repo

Clones the code into the (ephemeral) Colab runtime. Re-running this cell in a fresh
session re-clones; the *data and models* live on Drive, so nothing is lost.

In [ ]:
%cd /content
![ -d mh-repo ] && rm -rf mh-repo
!git clone https://github.com/SoshanW/mental-health-screening-fyp.git mh-repo
%cd /content/mh-repo

## 4. Install dependencies

Colab ships torch (CUDA) preinstalled, so we install only the rest of the modeling stack.

In [ ]:
# Colab ships torch (CUDA) preinstalled, so we install only the rest of the stack.
# transformers is pinned to the exact version the code is tested against locally
# (5.14.1), so the base_model/pooler_output behaviour is identical here and locally.
# cleanlab (section 13) is capped below 3.0 because the code calls the v2 API; its
# deps are pure-python (numpy/pandas/scikit-learn/termcolor/tqdm), so installing it
# cannot disturb the preinstalled CUDA torch.
!pip install -q 'transformers==5.14.1' 'accelerate>=0.26.0' 'scikit-learn>=1.4' 'cleanlab>=2.6,<3'
import torch, transformers, cleanlab
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cleanlab', cleanlab.__version__,
      '| CUDA available:', torch.cuda.is_available())

## 4b. Authenticate with Hugging Face (required — MentalBERT is gated)

`mental/mental-bert-base-uncased` is a **gated** repo, so downloading it needs a logged-in
token that has accepted the model terms. One-time: sign in to Hugging Face, open the
[model page](https://huggingface.co/mental/mental-bert-base-uncased) and click **Agree and
access repository**, then make a **read** token. Store it as a Colab Secret named
`HF_TOKEN` (key icon in the left sidebar → toggle notebook access) so the cell below logs
in automatically; otherwise it prompts you to paste the token. Re-run this cell after any
runtime reset.

In [ ]:
# MentalBERT (mental/mental-bert-base-uncased) is a GATED HF repo. One-time setup:
#   1. Sign in at huggingface.co.
#   2. Open https://huggingface.co/mental/mental-bert-base-uncased and click
#      "Agree and access repository".
#   3. Make a READ token at https://huggingface.co/settings/tokens.
# Recommended: store the token in Colab Secrets (key icon, left sidebar) as HF_TOKEN
# and enable notebook access -- then this cell logs in automatically every runtime.
# Otherwise the login() widget prompts you to paste it.
import os
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    login(token=token)
    print('Authenticated to Hugging Face via stored token.')
else:
    login()  # opens a widget; paste your READ token

# Sanity check: confirm this account can actually see the gated MentalBERT repo.
from huggingface_hub import HfApi
HfApi().model_info('mental/mental-bert-base-uncased')
print('Access to mental/mental-bert-base-uncased confirmed.')

## 5. Sanity-check the harmonized data load

Optional but recommended — confirms the loaders see your Drive data before training.

In [ ]:
!python -m src.data --data-root "$DATA_ROOT"

## 6. (Optional) Fast wiring smoke-test

Runs the full train→predict pipeline in seconds on a tiny public model and 50 samples —
verifies the plumbing before committing to a real MentalBERT run. Skip once you trust it.

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "/content/smoke_models" \
    --model-name hf-internal-testing/tiny-random-bert \
    --max-train-samples 50 --num-epochs 1
!python -m src.modeling.predict --artifacts-root "/content/smoke_models"

In [ ]:
# Fast first run: 1 epoch, isolated on its own Drive dir so it never collides with
# (or gets resumed into) the full 3-epoch run below. Train -> predict -> confusion
# matrix, end to end on the REAL MentalBERT. ~1 h on a T4.
# If you hit CUDA out-of-memory at --batch-size 32, drop it to 16.
FAST_ROOT = f'{DRIVE_ROOT}/Models_fast'

!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "$FAST_ROOT" \
    --num-epochs 1 --batch-size 32 --fp16 --save-steps 500 --resume
!python -m src.modeling.predict --artifacts-root "$FAST_ROOT"

import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred = pd.read_csv(f'{FAST_ROOT}/predictions/test_predictions.csv')
labels = sorted(set(pred['true_condition']) | set(pred['predicted_condition']))
cm = confusion_matrix(pred['true_condition'], pred['predicted_condition'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels],
                     columns=[f'pred_{l}' for l in labels])
print('=== fast-run confusion matrix (rows = truth, cols = prediction) ===')
print(cm_df.to_string())
print('\n=== fast-run per-condition report ===')
print(classification_report(pred['true_condition'], pred['predicted_condition'],
                            labels=labels, zero_division=0, digits=4))
cm_df

## 7. Train the real MentalBERT baseline

Checkpoints, splits, and predictions are written under `Models/` **on Drive**, so they
persist. `--fp16` speeds things up on the GPU. Adjust `--batch-size` up (e.g. 32) if the
GPU has spare memory (T4 ≈ 15 GB).

**Crash-safe / resumable.** `--save-steps 500` writes a checkpoint every 500 steps to
Drive (keeping the 2 most recent), and `--resume` picks up from the latest one. So if
the Colab session drops mid-run, **just re-run this cell** — it continues from the last
checkpoint instead of starting over. (Step-based saving disables best-model-at-end; the
final checkpoint is the last step's weights.)

This is the long one — roughly 2.5–4 h for 3 epochs on a T4. For a quick first pass, add
`--num-epochs 1` (and optionally `--max-length 128`) to get a checkpoint + confusion
matrix in ~1 h, then decide if the full run is worth it.

In [ ]:
!python -m src.modeling.train \
    --data-root "$DATA_ROOT" --artifacts-root "$ARTIFACTS_ROOT" \
    --num-epochs 3 --batch-size 16 --fp16 \
    --save-steps 500 --resume

## 8. Predict on the held-out test split

Reads `Models/splits/test.csv` and the trained checkpoint, writes
`Models/predictions/test_predictions.csv`, and prints accuracy + macro-F1.

In [ ]:
!python -m src.modeling.predict --artifacts-root "$ARTIFACTS_ROOT"

In [ ]:
import pandas as pd
pred = pd.read_csv(f'{ARTIFACTS_ROOT}/predictions/test_predictions.csv')
print('rows:', len(pred))
pred.head()

## 9. Per-condition confusion matrix + report

Builds the per-condition confusion matrix and precision/recall/F1 report directly
from `test_predictions.csv`. Only conditions actually present in the held-out test set
are shown (today: `bipolar`, `depression` — the other POC classes have no data yet).
This is the reference-point breakdown every later method (noise model → abstention)
must beat.

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred = pd.read_csv(f'{ARTIFACTS_ROOT}/predictions/test_predictions.csv')

# Label set = conditions that actually appear as a true label OR a prediction.
labels = sorted(set(pred['true_condition']) | set(pred['predicted_condition']))

cm = confusion_matrix(pred['true_condition'], pred['predicted_condition'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels],
                     columns=[f'pred_{l}' for l in labels])

print('=== Per-condition confusion matrix (rows = truth, cols = prediction) ===')
print(cm_df.to_string())
print('\n=== Per-condition classification report ===')
print(classification_report(pred['true_condition'], pred['predicted_condition'],
                            labels=labels, zero_division=0, digits=4))

# Persist alongside the predictions so later steps (C1 noise model) can reload it.
cm_out = f'{ARTIFACTS_ROOT}/predictions/confusion_matrix.csv'
cm_df.to_csv(cm_out)
print('Wrote confusion matrix ->', cm_out)
cm_df

## 10. C1 diagnostic — extract MentalBERT pooled embeddings (train split)

First step of the condition-dependent noise diagnostic (`src/noise/`, DECISIONS.md
D-030 / open item 2). This is **inference only** — no training. It loads the saved
checkpoint + tokenizer from `Models/checkpoints/latest` on Drive (so the gated-HF
login in step 4b is **not** needed here), runs each training-split post through the
model, and caches the pooled pre-classification-head representation
(`base_model.pooler_output`) to `Models/embeddings/train/` on Drive as
`features.npy` + `metadata.csv`.

`--output-dtype float16` keeps the artefact ~172 MB (cosine-neighbour ordering is
robust to it; local analysis upcasts to float32). A few minutes on a T4. The next
stage (`src/noise/clusterability.py`) runs **locally** on the cached embeddings — no
GPU needed — so bring `Models/embeddings/train/` back down (Drive Desktop sync or
download) and run `python -m src.noise.clusterability --embeddings-dir <path>`.

In [ ]:
# Extract + cache pooled embeddings for the TRAIN split. Inference only; writes
# Models/embeddings/train/{features.npy, metadata.csv} to Drive. ~172 MB at fp16.
# Point --checkpoint-dir elsewhere if your 3-epoch checkpoint is not at
# <ARTIFACTS_ROOT>/checkpoints/latest.
!python -m src.noise.embeddings \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --split train \
    --output-dtype float16 \
    --batch-size 64

In [ ]:
# Sanity-check the cached embeddings before pulling them down for local analysis.
import numpy as np, pandas as pd

emb_dir = f'{ARTIFACTS_ROOT}/embeddings/train'
feats = np.load(f'{emb_dir}/features.npy')
meta = pd.read_csv(f'{emb_dir}/metadata.csv')

print('shape', feats.shape, '| dtype', feats.dtype,
      '| finite', bool(np.isfinite(feats).all()),
      '| aligned', len(feats) == len(meta))
# Expect ~112k rows, no daic_woz, depression-heavy skew. If counts look like the full
# ~140k or include daic_woz, the wrong split was embedded.
print('\nsource(s):', sorted(meta['source'].unique()))
print(meta['condition'].value_counts())

## 10b. D-035 control — base MentalBERT embeddings

The clusterability diagnostic on the fine-tuned embeddings (§10) came back with high
agreement for every condition, including bipolar (DECISIONS.md **D-034**: the
prediction was falsified). One confound is that those embeddings were fine-tuned to
separate these same noisy labels, so the structure may be a training artifact rather
than intrinsic to the text.

This control re-extracts using **base `mental/mental-bert-base-uncased`** (no
fine-tuning on our labels). The **delta** between fine-tuned and base agreement
quantifies how much of §10's structure is artifact. It does **not** produce a "true"
clusterability number (true labels are unavailable).

**The extract cell below needs the step 4b HF login** (base MentalBERT is gated). It
writes to a distinct Drive path so §10's cache is preserved. The comparison cell after
it runs the diagnostic (torch-free, no GPU) on **both** caches straight from Drive, on
the same protocol as D-034 (seed 0, |E| = 15000, G = 20), and prints the
`delta = finetuned - base` table. You can also run that same command in your local
`.venv` after syncing the two folders down; the result is identical (deterministic).

**Interpretation rule, set in advance (D-035):** base bipolar agreement above ~70%
means the clusterability-failure argument is genuinely weak and should be dropped;
below ~40% means §10's number was substantially artifact. HOC itself (Stage 2) still
runs on the **fine-tuned** embeddings, matching HOC's own protocol, not on these.

In [ ]:
# D-035 control: extract embeddings from BASE MentalBERT (NO fine-tuning on our four
# labels). REQUIRES the HF login in step 4b -- base MentalBERT is a gated repo (the
# fine-tuned run in step 10 did not need it, because it loads a local checkpoint).
# Writes to a DISTINCT path Models/embeddings/train__base/ so the fine-tuned cache is
# never overwritten. ~172 MB at fp16, a few minutes on a T4.
!python -m src.noise.embeddings \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --extractor base \
    --split train \
    --output-dtype float16 \
    --batch-size 64

In [ ]:
# D-035 comparison: run the clusterability diagnostic on BOTH caches (fine-tuned vs
# base) straight from Drive, same protocol as D-034 (seed 0, |E|=15000, G=20).
# Torch-free; prints both reports + the delta table and writes the CSVs to Drive.
# (Identical if run in the local .venv instead.)
!python -m src.noise.clusterability \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train" \
    --base-embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__base" \
    --seed 0 --sample-size 15000 --n-rounds 20

## 11. Stage 2 — HOC transition-matrix estimator

Runs the HOC estimator (Zhu, Song & Liu 2021) on the **fine-tuned** embeddings from
section 10. Fine-tuned, **not** base: HOC's protocol takes the extractor from a model
trained to near-100% accuracy on the noisy labels, so base features would handicap it
unfairly (DECISIONS.md D-035). It estimates a 4x4 transition matrix `T`
(`T[i][j] = P(noisy=j | true=i)`, rows/cols in the order bipolar, depression,
eating_disorder, schizophrenia) and the true-class prior `p`.

It runs under **5 random seeds** and reports the **spread** (per-element std), not a
point estimate: seed-instability is itself the finding (D-030/D-036). For each seed it
also flags whether each row is diagonally dominant (Assumption 2), whether `T` is
non-singular (Assumption 1), and how the estimated prior compares to the observed noisy
proportions.

**Read D-036's pre-registered prediction before you look at the output.** The prediction
(all diagonals >= 0.85, because fine-tuning suppressed the disagreement signal HOC needs)
and its falsifier (bipolar diagonal < 0.7 with a dominant bipolar-to-depression
off-diagonal, stable across seeds -> HOC works, drop the clusterability argument) were
fixed in advance. The cell prints the raw matrix first, then whether the prediction held.
Do not fit the interpretation to the result.

Prerequisites: sections 2, 3, 4 (mount, clone, install) and the section-10 fine-tuned
cache. The gated-HF login (4b) is **not** needed here (it reads cached embeddings, not a
gated model). Runs on CPU in a few minutes; a few more with more seeds.

In [ ]:
# Stage 2: HOC transition-matrix estimator on the FINE-TUNED embeddings (NOT base,
# per D-035). Runs >=5 seeds and reports the spread, per-row diagonal dominance
# (HOC Assumption 2), non-singularity (Assumption 1), and estimated prior vs noisy
# marginal. Torch-dependent (Adam) but runs on the cached embeddings, so CPU is fine.
# G=50, |E|=15000, max_iter=1500 = HOC's "Global" setting. Reads the fine-tuned cache
# from section 10. Read DECISIONS.md D-036's pre-registered prediction before you read
# the output; the cell prints the raw result first, then whether the prediction held.
!python -m src.noise.hoc_estimate \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train" \
    --seeds 0 1 2 3 4 \
    --n-rounds 50 --sample-size 15000 --max-iter 1500

## 12. C1 second estimator — out-of-sample probabilities (author-grouped CV)

The second independent noise estimator is cleanlab (Confident Learning), and it needs
**out-of-sample** predicted probabilities: every post's probability must come from a
model that never trained on it. This cell produces them by **3-fold cross-validation on
the training split**, fine-tuning MentalBERT once per fold.

**Non-negotiable (D-010):** folds are `GroupKFold` on `author_id`. A random KFold would
not error but would leak an author's writing style across the fit/predict boundary and
silently contaminate every downstream cleanlab number. (A unit test asserts no author
appears on both sides of any fold.)

**This is the expensive GPU step** and it needs the **§4b HF login** (it fine-tunes from
the gated base MentalBERT). It reuses the Milestone 0 hyperparameters so the folds match
the baseline. It is **resumable per fold** — if the session drops, just re-run the cell
and it skips folds already cached in `Models/oos/train/`. Reads `Models/splits/train.csv`.

**Why there is no progress bar.** 3 folds × 3 epochs ≈ 40 000 training steps, and tqdm
redraws its bar on every step. Piped into one Colab output cell, those redraws accumulate
in the page until the browser tab freezes — the training keeps going, but the notebook
becomes unusable. So this step runs with tqdm and the HF banners off, printing a
timestamped line per fold plus a loss line every `--logging-steps` steps (a few hundred
lines for the whole job), and tees everything to `Models/oos/train/run.log` on Drive.
**Sparse output is expected — it is not a hang.** If the tab has already frozen, kill it
and re-run the cell: completed folds reload from cache. To watch progress from a second
notebook (or after a disconnect), tail the log:
`!tail -n 40 "$ARTIFACTS_ROOT/oos/train/run.log"`.

**Note the D-011 caveat:** the code has no `WeightedLossTrainer` (that documented class
was never implemented); the baseline and these folds both use a plain unweighted
`Trainer`. Prerequisites: §2, §3, §4, §4b.

In [ ]:
# Stage 3a: out-of-sample probabilities via AUTHOR-GROUPED 3-fold CV (for cleanlab).
# GPU + the section-4b HF login are required: this FINE-TUNES MentalBERT k=3 times
# (once per fold) from the gated base model, reusing the Milestone 0 hyperparameters
# so the folds are comparable to the baseline. Folds are GroupKFold on author_id
# (D-010) -- random folds would silently leak writing style and contaminate every
# downstream cleanlab number. Resumable per fold: a dropped session re-run skips
# completed folds (cached in Models/oos/train/). Reads Models/splits/train.csv.
#
# OUTPUT IS DELIBERATELY SPARSE. 3 folds x 3 epochs is ~40k training steps, and a
# tqdm bar redraws on every one of them; streamed into one Colab output cell that
# is what freezes the browser tab (the run itself is fine). So the CLI disables
# tqdm by default and prints one line per --logging-steps steps, and everything is
# tee'd to a log on Drive that survives a dropped session. Expect a heartbeat every
# few minutes, not a moving bar. Pass --progress-bars only in a real terminal.
#
# COST: ~6 h on a T4 at 3 epochs/fold. Add `--num-epochs 1` to cut it to ~2 h --
# the baseline found 1 epoch ~= 3 epochs (label-noise ceiling), so OOS probs barely
# change. Check the cost against your Colab quota before launching.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # also quiets the child process
OOS_LOG = f'{ARTIFACTS_ROOT}/oos/train/run.log'
os.makedirs(os.path.dirname(OOS_LOG), exist_ok=True)
print('logging to', OOS_LOG)

# `-u` = unbuffered, so the heartbeat lines appear live instead of in one dump.
!python -u -m src.noise.oos_probabilities \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --split train \
    --n-folds 3 \
    --fp16 \
    --logging-steps 500 2>&1 | tee -a "$OOS_LOG"

## 13. C1 second estimator — cleanlab (Confident Learning) transition matrix

Runs cleanlab v2 on the §12 out-of-sample probabilities and the noisy proxy labels,
producing a 4×4 transition matrix in HOC's orientation (`T[i][j] = P(noisy=j | true=i)`,
rows sum to 1), same sorted order as §11 (bipolar, depression, eating_disorder,
schizophrenia). **Torch-free — runs on CPU, no HF login.**

It prints the cleanlab matrix **beside the §11 HOC mean matrix, elementwise**, and
answers the D-037 question explicitly: does cleanlab **also** return a near-identity
bipolar row (agreeing with HOC that there is almost no estimated noise), or does it
disagree? If two independent estimators fail identically, that is the strongest form of
the C1 result. The output is raw — not smoothed or clipped — because both outcomes are
informative.

Prerequisites: §12 (`Models/oos/train/`) and §11 (`Models/embeddings/train/hoc_mean_T.csv`).
This step needs neither GPU nor the HF login; you can also run the same command in your
local `.venv` after syncing `Models/oos/train/` down.

In [ ]:
# Stage 3b: cleanlab (Confident Learning) transition matrix on the section-12 OOS
# probabilities. Torch-free -> runs on CPU, no HF login needed. Prints the cleanlab
# 4x4 matrix beside the section-11 HOC mean matrix elementwise and answers the D-037
# question: does cleanlab ALSO return a near-identity bipolar row (agreeing with HOC
# that there is almost no estimated noise), or does it disagree? Reported raw.
# Needs Models/oos/train/ (section 12) and Models/embeddings/train/hoc_mean_T.csv (section 11).

# Self-contained guard: section 4 installs cleanlab, but this section is documented as
# runnable on its own (no GPU, no HF login), and a runtime that skipped section 4 -- or
# was reset after section 12's long run -- would otherwise fail here with
# ModuleNotFoundError only AFTER re-reading the artefacts. Cheap no-op when present.
try:
    import cleanlab
    print('cleanlab', cleanlab.__version__)
except ModuleNotFoundError:
    !pip install -q 'cleanlab>=2.6,<3'
    import cleanlab
    print('installed cleanlab', cleanlab.__version__)

!python -m src.noise.cleanlab_estimate \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --split train

## 14. D-040: HOC on representations NOT trained on our labels

Liu, Cheng & Zhang (2023, ICML) document **four** identifiability routes, not three.
The fourth is **disentangled informative features** (their Theorem 5.5), and their
Table 1 estimates T on CIFAR-10 with HOC over three encoders: weakly supervised
(14.51 error), SimCLR (4.42), IPIRM (3.73). The worst row is the standard protocol,
which is what §11 ran. **DECISIONS.md D-040** pre-registers whether the §11 result
survives that remedy, and fixes the thresholds *before* these cells are run.

**Read D-040 before reading any output from this section.** It states the numeric
prediction (bipolar diagonal falls below 0.70 on Arm A, most likely 0.05 to 0.45) and a
five-branch decision rule. The rule is evaluated automatically by `--d040-arm`, so
the branch is read off the data rather than chosen afterwards.

| Arm | Extractor | Cache | Needs |
|---|---|---|---|
| A | base MentalBERT | `embeddings/train__base` | already exists from §10b |
| B | `sentence-transformers/all-mpnet-base-v2` | `embeddings/train__mpnet` | GPU, no HF login |

D-035 is **not** reversed by this section: §11 remains the fair test of
HOC-as-published, and these arms answer a different question. Both arms reuse §11's
exact configuration (seeds `0 1 2 3 4`, G=50, |E|=15000, max_iter=1500, lr=0.1); a
configuration difference between arms would make the comparison uninterpretable.

In [ ]:
# === 14a. BOTH D-040 ARMS, END TO END, IN ONE CELL ==========================
# Everything section 14 needs: features -> 2-NN agreement -> HOC, for Arm A (base
# MentalBERT) and Arm B (all-mpnet-base-v2). The cells below this one are the same
# steps broken out individually; use them to re-run a single stage. Run the section
# 14 summary cell afterwards for the D-041 table (per-arm diagonals, the HOC-vs-2NN
# Pearson coupling, and the per-seed spread).
#
# BEFORE YOU RUN, read DECISIONS.md D-040. It fixes the prediction and the decision
# thresholds in advance, and reading the numbers first defeats the point of that.
#
# Requirements:
#   * GPU runtime (Arm B extraction). The HOC and clusterability stages are CPU-bound.
#   * Section 4b HF login ONLY if Arm A's cache is missing: base MentalBERT is gated.
#     Arm B's encoder is not gated. If section 10b already ran, Arm A is reused as-is.
#   * Section 14's parity cell is worth running once before trusting Arm B; it checks
#     our mean pooling against SentenceTransformer.encode() on the real weights.
#
# Expect roughly 1 to 1.5 hours in total, dominated by HOC: each arm does 5 seeds x 50
# rounds of brute-force cosine 2-NN over |E| = 15000, on Colab's small CPU allocation.
# RESUME = True makes the cell safe to re-run after a session drop: any stage whose
# output already exists is skipped rather than repeated.

RESUME = True

import subprocess, sys, time
from pathlib import Path

ROOT = Path(ARTIFACTS_ROOT)
ARMS = {
    'base':  {'dir': ROOT / 'embeddings' / 'train__base',
              'gated': True,
              'note': 'Arm A: base MentalBERT (D-035/D-036 control cache)'},
    'mpnet': {'dir': ROOT / 'embeddings' / 'train__mpnet',
              'gated': False,
              'note': 'Arm B: sentence-transformers/all-mpnet-base-v2'},
}
# D-037's configuration, reused verbatim. Do NOT edit: an arm-to-arm configuration
# difference would make the whole D-040 comparison uninterpretable (see D-040).
SEEDS = ['0', '1', '2', '3', '4']
HOC_CFG = ['--n-rounds', '50', '--sample-size', '15000', '--max-iter', '1500', '--lr', '0.1']

try:
    import torch
    print('GPU visible:', torch.cuda.is_available())
except ImportError:
    print('torch not importable; run section 4 first.')


def run(stage, *args):
    """Run a repo module as a subprocess, streaming its output, failing loudly."""
    print(f'\n{"=" * 74}\n{stage}\n{"=" * 74}', flush=True)
    t0 = time.time()
    subprocess.run([sys.executable, '-m', *args], check=True)
    print(f'[done in {(time.time() - t0) / 60:.1f} min]', flush=True)


def have_features(d):
    return (d / 'features.npy').exists() and (d / 'metadata.csv').exists()


t_start = time.time()

# --- Stage 1: features ------------------------------------------------------
for name, arm in ARMS.items():
    if RESUME and have_features(arm['dir']):
        print(f"SKIP extraction [{name}]: cache already at {arm['dir']}")
        continue
    if arm['gated']:
        print(f"NOTE [{name}]: gated repo, this needs the section 4b HF login.")
    run(f"EXTRACT [{name}] {arm['note']}",
        'src.noise.embeddings',
        '--artifacts-root', str(ROOT),
        '--extractor', name,
        '--split', 'train',
        '--output-dtype', 'float16',
        '--batch-size', '64')

# --- Stage 2: 2-NN agreement (D-041 needs it per arm, for the coupling) -----
for name, arm in ARMS.items():
    out = arm['dir'] / 'clusterability_report_within_e.csv'
    if RESUME and out.exists():
        print(f'SKIP clusterability [{name}]: {out.name} exists')
        continue
    run(f'CLUSTERABILITY [{name}] 2-NN agreement, within_e, same protocol as D-034/D-036',
        'src.noise.clusterability',
        '--embeddings-dir', str(arm['dir']),
        '--scope', 'within_e')

# --- Stage 3: HOC, with the pre-registered rule evaluated per arm -----------
for name, arm in ARMS.items():
    out = arm['dir'] / 'hoc_d040_verdict.csv'
    if RESUME and out.exists():
        print(f'SKIP HOC [{name}]: {out.name} exists (delete it to force a re-run)')
        continue
    run(f'HOC [{name}] 5 seeds, D-037 configuration, --d040-arm',
        'src.noise.hoc_estimate',
        '--embeddings-dir', str(arm['dir']),
        '--d040-arm',
        '--seeds', *SEEDS,
        *HOC_CFG)

print(f'\n{"=" * 74}\nALL D-040 ARMS COMPLETE in {(time.time() - t_start) / 60:.1f} min'
      f'\n{"=" * 74}')

# --- Headline: the pre-registered branch per arm ----------------------------
import pandas as pd

for name, arm in ARMS.items():
    v = arm['dir'] / 'hoc_d040_verdict.csv'
    if not v.exists():
        print(f'{name}: no verdict written'); continue
    r = pd.read_csv(v).iloc[0]
    print(f"  {name:>6}: branch={r['branch']:<18} bipolar diag={r['bipolar_diagonal']:.4f} "
          f"+/- {r['bipolar_diagonal_std']:.4f}   mean |det|={r['mean_abs_det']:.4f}")

print('\nSTRENGTHENED needs EVERY arm at STRENGTHENED_ARM. One arm >= 0.85 with another '
      '< 0.70 is DISAGREE, and D-040 says that is reported as the outcome, not resolved '
      'by picking an arm. Run the section 14 summary cell for the full D-041 table.')


In [ ]:
# D-040 Arm B: extract embeddings from a contrastively trained SENTENCE ENCODER
# (sentence-transformers/all-mpnet-base-v2) that has never seen our four labels and
# is not a mental-health model at all. NOT a gated repo, so no HF login is needed.
# Writes to a DISTINCT path Models/embeddings/train__mpnet/, so neither the §10
# fine-tuned cache nor the §10b base cache can be clobbered. ~172 MB at fp16.
#
# Two things this run holds fixed on purpose (see DECISIONS.md D-040):
#   * --max-length stays at the project-wide 256 cap (methodology 3.4.3), even though
#     all-mpnet-base-v2's own max_seq_length is 384, so a difference between arms can
#     never be attributed to the arms having read different amounts of each post.
#   * the feature is masked mean pooling over last_hidden_state, then L2-normalise,
#     which is this encoder's published pooling. It is NOT pooler_output (§10's
#     feature): mpnet's [CLS] pooler was never trained by the sentence objective.
!python -m src.noise.embeddings \
    --artifacts-root "$ARTIFACTS_ROOT" \
    --extractor mpnet \
    --split train \
    --output-dtype float16 \
    --batch-size 64


In [ ]:
# OPTIONAL but recommended once: confirm our masked mean pooling reproduces the
# published encoder. src/noise/embeddings.py reimplements all-mpnet-base-v2's pooling
# (mean over non-padding positions, then L2 normalise) rather than calling
# SentenceTransformer.encode(), because encode() owns its own tokenisation and would
# silently apply a 384-token window instead of the project's 256 cap. This cell checks
# the reimplementation against the real thing on a handful of SHORT posts, where the
# two truncation limits cannot diverge, so any mismatch is a pooling bug.
try:
    import sentence_transformers
except ModuleNotFoundError:
    !pip install -q 'sentence-transformers>=5.0,<6'
    import sentence_transformers
print('sentence-transformers', sentence_transformers.__version__)

import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from src.noise.embeddings import SENTENCE_MODEL_ID, EmbeddingConfig, extract_mean_pooled_embeddings

# Short posts only: under ~200 words both paths see the whole text untruncated.
train = pd.read_csv(f'{ARTIFACTS_ROOT}/splits/train.csv')
short = train[train['text'].astype(str).str.split().str.len().between(5, 150)].head(32)

ours = extract_mean_pooled_embeddings(short, SENTENCE_MODEL_ID, EmbeddingConfig(batch_size=8))
theirs = SentenceTransformer(SENTENCE_MODEL_ID).encode(
    short['text'].astype(str).tolist(), normalize_embeddings=True, batch_size=8
)

cos = (ours.astype(np.float32) * np.asarray(theirs, dtype=np.float32)).sum(axis=1)
print(f'per-post cosine vs SentenceTransformer.encode(): min={cos.min():.6f} '
      f'mean={cos.mean():.6f}')
print('PARITY OK' if cos.min() > 0.999 else
      'MISMATCH -- do NOT run Arm B until this is understood (check the pooling).')


In [ ]:
# Sanity-check the Arm B cache before using it, mirroring §10's check.
import numpy as np, pandas as pd

emb_dir = f'{ARTIFACTS_ROOT}/embeddings/train__mpnet'
feats = np.load(f'{emb_dir}/features.npy')
meta = pd.read_csv(f'{emb_dir}/metadata.csv')

print('shape', feats.shape, '| dtype', feats.dtype)
print('extractor stamp:', meta['extractor'].unique())      # must be exactly ['mpnet']
print('rows aligned:', len(feats) == len(meta))
print('condition counts:\n', meta['condition'].value_counts())
# Vectors are L2-normalised by the extractor, so every norm should be ~1.0.
norms = np.linalg.norm(feats.astype(np.float32), axis=1)
print(f'L2 norms: min={norms.min():.4f} max={norms.max():.4f}')


In [ ]:
# 2-NN agreement on the Arm B features, same protocol as D-034/D-036 (seed 0,
# |E|=15000, G=20, within_e). D-041 needs this per arm: D-037 recorded that HOC's
# diagonal tracked this statistic at Pearson 0.999 on the fine-tuned features, and
# whether that coupling persists on a different representation is itself a finding.
# Torch-free, so this is equally runnable in the local .venv.
!python -m src.noise.clusterability \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__mpnet" \
    --scope within_e

# Arm A's number already exists from §10b, but re-run it here so both arms' reports
# land in the same format and scope that the correlation cell below reads.
!python -m src.noise.clusterability \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__base" \
    --scope within_e


In [ ]:
# D-040 ARM A: HOC on the BASE MentalBERT cache from §10b. No new training and no new
# extraction; this is a pure re-run of §11's estimator against different features.
# --d040-arm declares the intent, which (a) switches the non-fine-tuned warning from
# "you are substituting for the D-037 protocol" to "this is the sanctioned arm", and
# (b) evaluates and PERSISTS the D-040 pre-registered decision rule alongside the
# result, so the branch cannot be chosen after the fact.
# Config is byte-for-byte §11's. Do not change it: an arm-to-arm config difference
# would make the whole comparison uninterpretable.
!python -m src.noise.hoc_estimate \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__base" \
    --d040-arm \
    --seeds 0 1 2 3 4 \
    --n-rounds 50 --sample-size 15000 --max-iter 1500 --lr 0.1


In [ ]:
# D-040 ARM B: the same estimator, the same configuration, on the sentence-encoder
# features. Torch-dependent (Adam) but runs off the cached embeddings, so CPU is fine.
!python -m src.noise.hoc_estimate \
    --embeddings-dir "$ARTIFACTS_ROOT/embeddings/train__mpnet" \
    --d040-arm \
    --seeds 0 1 2 3 4 \
    --n-rounds 50 --sample-size 15000 --max-iter 1500 --lr 0.1


In [ ]:
# D-041 inputs: per-arm diagonal comparison, the HOC-vs-2NN coupling, and the branch.
# D-037 measured that coupling at Pearson 0.999 on the fine-tuned features, which is
# what established that HOC's output there was a deterministic function of a statistic
# D-036 had already shown to be largely a fine-tuning artifact. Whether it persists on
# a representation supervised differently is a finding in its own right.
import numpy as np, pandas as pd
from pathlib import Path

ARMS = {'finetuned': 'embeddings/train',
        'base': 'embeddings/train__base',
        'mpnet': 'embeddings/train__mpnet'}
CONDS = ['bipolar', 'depression', 'eating_disorder', 'schizophrenia']

rows, per_arm = [], {}
for arm, rel in ARMS.items():
    d = Path(ARTIFACTS_ROOT) / rel
    t_path = d / 'hoc_mean_T.csv'
    # The within_e report is what the current CLI writes; the fine-tuned arm predates
    # that filename and has the same numbers under the older one.
    c_path = next((p for p in (d / 'clusterability_report_within_e.csv',
                               d / 'clusterability_report.csv') if p.exists()),
                  d / 'clusterability_report_within_e.csv')
    if not t_path.exists():
        print(f'skip {arm}: no {t_path.name} yet'); continue
    t = pd.read_csv(t_path, index_col=0).reindex(
        index=[f'true_{c}' for c in CONDS], columns=[f'noisy_{c}' for c in CONDS])
    diag = np.diag(t.to_numpy(dtype=float))
    per_arm[arm] = diag

    coupling = np.nan
    if c_path.exists():
        rep = pd.read_csv(c_path).set_index('condition').reindex(CONDS)
        coupling = float(np.corrcoef(diag, rep['per_neighbor_agreement'].to_numpy())[0, 1])
    else:
        print(f'note: {arm} has no {c_path.name}; run the clusterability cell for the coupling')

    ps = pd.read_csv(d / 'hoc_per_seed.csv') if (d / 'hoc_per_seed.csv').exists() else None
    rows.append({
        'arm': arm,
        **{f'diag_{c}': v for c, v in zip(CONDS, diag)},
        'bipolar_seed_std': float(ps['diag_bipolar'].std(ddof=0)) if ps is not None else np.nan,
        'mean_abs_det': float(ps['det'].abs().mean()) if ps is not None else np.nan,
        'pearson_diag_vs_2nn': coupling,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

for arm in ('base', 'mpnet'):
    v = Path(ARTIFACTS_ROOT) / ARMS[arm] / 'hoc_d040_verdict.csv'
    if v.exists():
        row = pd.read_csv(v).iloc[0]
        print(f"\nD-040 branch [{arm}]: {row['branch']}  "
              f"(bipolar diag {row['bipolar_diagonal']:.4f}, "
              f"mean |det| {row['mean_abs_det']:.4f})")

print('\nCaveat for D-041 (D-036): sample_size is fixed at 15000 in every arm, so the '
      'neighbour starvation is IDENTICAL across arms and does not invalidate the '
      'comparison, but it bounds what any arm can show. At |E|=15000 a bipolar post has '
      '~566 same-class candidates versus ~4221 in the full split.')
